# 파이썬 업무자동화 ② API 자동화

> **실습 준비**
>
> - 이 노트북 파일의 코드는 인터넷으로 데이터를 받아 옵니다. 실습 전에 인터넷 연결을 확인합니다.
> - 날씨와 환율 값은 실행하는 날마다 달라집니다. 실행 결과에 적힌 숫자는 예시입니다.
> - 교안의 코드는 `r"C:\workspace\python20261103\6일차"`처럼 전체 경로를 씁니다. 노트북에서는 노트북 파일이 있는 폴더를 기준으로 저장합니다.

# 실습 2. API 자동화

## API 자동화란?

웹 서비스에 데이터를 요청하고, 받은 값을 업무 파일로 남깁니다.

| 단계 | 하는 일 | 예시 → 처리 결과 |
|---|---|---|
| 01 요청 보내기 | 주소로 데이터 요청 | `requests.get(url)` → `<Response [200]>` |
| 02 JSON 받기 | 딕셔너리로 바꾸기 | `response.json()` → `{"daily": {...}}` |
| 03 값 꺼내기 | 키·순번으로 고르기 | `daily["time"][1]` → `"2026-11-09"` |
| 04 파일로 저장 | CSV·엑셀로 남기기 | `writer.writerow(행)` → 날씨.csv |

## 도구: requests · json · csv · openpyxl

| 모듈 | 역할 | 자주 쓰는 기능 |
|---|---|---|
| requests | 웹 주소로 데이터 요청 | `requests.get()`, `.status_code`, `.raise_for_status()`, `.json()` |
| json | JSON 구조 보기 좋게 출력 | `json.dumps(data, indent=2, ensure_ascii=False)` |
| csv | CSV 파일로 저장 | `csv.writer()`, `.writerow()` |
| openpyxl | 엑셀 파일 만들기·저장 | `Workbook()`, `ws.append()`, `wb.save()` |

- requests·openpyxl은 설치 필요 → 터미널에서 `pip install requests openpyxl`
- json·csv는 파이썬에 기본 포함 → 설치 없이 import만 하면 됩니다.

주피터 노트북에서는 아래 셀로 설치할 수 있습니다. 이미 설치되어 있으면 건너뜁니다.

In [ ]:
%pip install requests openpyxl

## 기본 사용법 ① 요청 보내기

**목표** : 웹 주소로 데이터를 요청하고 응답 확인하기

| 기능 | 설명 |
|---|---|
| `requests.get(url)` | 주소로 데이터 요청하기 |
| `timeout=10` | 10초 안에 응답이 없으면 멈추기 |
| `status_code` | 200 성공 · 404 주소 없음 · 500 서버 오류 |

**주의** : 인터넷이 끊겨 있으면 ConnectionError가 납니다. 실습 전 연결을 확인하세요.

In [ ]:
import requests
url = "https://open.er-api.com/v6/latest/USD"
response = requests.get(url, timeout=10)  # 요청 보내기
print(response.status_code)               # 200이면 성공
response.raise_for_status()               # 실패면 여기서 멈춤

실행 결과
```
200
```

## 기본 사용법 ② JSON으로 바꾸기

**목표** : 응답을 딕셔너리로 바꿔 구조 확인하기

응답 구조 (일부)
```
{
  "result": "success",
  "base_code": "USD",
  "rates": {
    "USD": 1,
    "KRW": 1380.5,
    "JPY": 152.3,
    ...
  }
}
```

**주의** : 환율 값은 날마다 달라집니다. 화면의 숫자는 예시입니다.

In [ ]:
import requests
url = "https://open.er-api.com/v6/latest/USD"
response = requests.get(url, timeout=10)
data = response.json()     # JSON → 딕셔너리
print(type(data))          # 자료형 확인
print(data["result"])      # 키로 바로 꺼내기

실행 결과
```
<class 'dict'>
success
```

응답 전체 구조를 보고 싶으면 json 모듈로 보기 좋게 출력합니다. 내용이 길어서 앞부분만 출력합니다.

In [ ]:
import json
print(json.dumps(data, indent=2, ensure_ascii=False)[:500])

## 기본 사용법 ③ 값 꺼내기

**목표** : 키를 따라 들어가 원하는 값만 꺼내기

| 기능 | 설명 |
|---|---|
| `data["rates"]` | 키로 안쪽 딕셔너리 꺼내기 |
| `rates["KRW"]` | 한 번 더 키로 들어가 값 꺼내기 |
| `for code in [...]` | 여러 값을 반복해서 꺼내기 |

**주의** : 없는 키를 쓰면 KeyError가 납니다. 대소문자까지 똑같이 써야 합니다.

In [ ]:
# ①·② 코드에 이어서
data = response.json()
rates = data["rates"]              # 안쪽 딕셔너리
print(rates["KRW"])                # 1달러 = 몇 원
for code in ["KRW", "JPY", "EUR"]:  # 여러 통화 반복
    print(code, rates[code])

실행 결과 (숫자는 예시)
```
1380.5
KRW 1380.5
JPY 152.3
EUR 0.92
```

## 기본 사용법 ④ 파일로 저장

**목표** : 꺼낸 값을 CSV·엑셀 파일로 남기기

교안은 `6일차\환율` 폴더에 저장합니다. 노트북에서는 노트북 파일이 있는 폴더 아래에 `환율` 폴더를 먼저 만든 뒤 저장합니다.

**주의** : `encoding="utf-8-sig"`를 빼면 엑셀에서 CSV 한글이 깨집니다.

**CSV로 저장**

In [ ]:
import csv
from pathlib import Path
# ①~③ 코드에 이어서 (rates 준비됨)
# 교안: folder = r"C:\workspace\python20261103\6일차\환율"
folder = Path("환율")
folder.mkdir(exist_ok=True)       # 환율 폴더가 없으면 만들기
with open(folder / "환율.csv", "w", newline="",
          encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["통화", "환율"])    # 머리글
    for code in ["KRW", "JPY", "EUR"]:
        writer.writerow([code, rates[code]])

**엑셀로 저장하려면**

In [ ]:
from openpyxl import Workbook
wb = Workbook()
ws = wb.active
ws.append(["통화", "환율"])
for code in ["KRW", "JPY"]:
    ws.append([code, rates[code]])
wb.save(folder / "환율.xlsx")

실행 후 `환율` 폴더에 `환율.csv`와 `환율.xlsx`가 생깁니다. 엑셀로 열어 확인합니다.

## 예제 ① 내일 날씨 알림

**자동화 전: 매일 아침 직접 확인**
1. 날씨 사이트 접속
2. 지역 검색(서울)
3. 내일 최고·최저기온 확인
4. 강수확률 확인
5. 팀 메신저에 직접 입력

⋯ 매일 아침 반복

**자동화 후: 실행 한 번으로 알림 완성**
```
날짜: 2026-11-09
최고기온: 15.2℃
최저기온: 6.8℃
최대 강수확률: 60%
우산을 챙기세요.
외투를 준비하세요.
```

**자동화 규칙** : 서울 좌표로 내일 예보 요청 → 기온·강수확률 꺼내기 → 기준에 맞으면 알림 문구 출력

### 예제 ① 사용할 API: Open-Meteo

**Open-Meteo** : 키 없이 쓰는 무료 날씨 API → 위치와 받을 항목만 넘기면 됩니다

```python
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 37.5665,           # 위도 (서울)
    "longitude": 126.9780,         # 경도
    "daily": ["temperature_2m_max"],  # 받을 항목
    "timezone": "Asia/Seoul",      # 한국 시간 기준
    "forecast_days": 2             # 오늘·내일
}
```

**자주 쓰는 daily 항목**

| 항목 | 뜻 |
|---|---|
| temperature_2m_max | 최고기온 (℃) |
| temperature_2m_min | 최저기온 (℃) |
| precipitation_probability_max | 최대 강수확률 (%) |
| precipitation_sum | 강수량 (mm) |

**응답 구조 (일부)**
```
"daily": {"time": ["2026-11-08", "2026-11-09"],
          "temperature_2m_max": [14.1, 15.2]}
```

**참고** : 항목 이름은 공식 문서(open-meteo.com/en/docs)에서 체크하면 요청 주소가 자동으로 만들어집니다.

### 예제 ① 풀이: 단계별로 생각하기

- **입력** : 서울 좌표·예보 항목
- **처리** : 내일 예보를 받아 기온·강수확률 꺼내기
- **출력** : 내일 날씨 알림 문구

| 단계 | 하는 일 | 코드 |
|---|---|---|
| 1. 요청 조건 정하기 | 위치·예보 항목·기간을 params에 담기 | `params = {"latitude": 37.5665, "longitude": 126.9780, ...}` |
| 2. 요청 보내고 JSON 받기 | 주소 뒤 `?latitude=...` 는 requests가 붙여 줌 | `response = requests.get(url, params=params)`<br>`data = response.json()` |
| 3. 내일 값 꺼내기 | 리스트 순번 `[0]`은 오늘, `[1]`은 내일 | `daily = data["daily"]`<br>`max_temp = daily["temperature_2m_max"][1]` |
| 4. 기준에 맞으면 알림 출력 | 강수확률 50% 이상 → 우산, 최저 10℃ 미만 → 외투 | `if rain >= 50:`<br>`    print("우산을 챙기세요.")` |

### 예제 ① 풀이: 완성 코드

In [ ]:
import requests
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 37.5665,         # 서울 위도
    "longitude": 126.9780,       # 서울 경도
    "daily": [                   # 받을 항목
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_probability_max",
        "precipitation_sum"      # 강수량(mm)
    ],
    "timezone": "Asia/Seoul",
    "forecast_days": 2           # 오늘·내일
}
response = requests.get(url, params=params,
                        timeout=10)
response.raise_for_status()  # 실패하면 멈춤
daily = response.json()["daily"]
date = daily["time"][1]      # [1] = 내일
max_temp = daily["temperature_2m_max"][1]
min_temp = daily["temperature_2m_min"][1]
rain = daily["precipitation_probability_max"][1]
rain_mm = daily["precipitation_sum"][1]
print(f"날짜: {date}")
print(f"최고기온: {max_temp}℃")
print(f"최저기온: {min_temp}℃")
print(f"최대 강수확률: {rain}%")
print(f"강수량: {rain_mm}mm")
if rain is not None and rain >= 50:
    print("우산을 챙기세요.")
if min_temp < 10:
    print("외투를 준비하세요.")

실행 결과 (숫자는 예시)
```
날짜: 2026-11-09
최고기온: 15.2℃
최저기온: 6.8℃
최대 강수확률: 60%
강수량: 3.2mm
우산을 챙기세요.
외투를 준비하세요.
```

## 실습 미션: 일주일 예보를 CSV로 저장하기

**미션** : 예제 ① 코드를 고쳐서, 7일치 예보를 주간날씨.csv로 저장해 보세요.

**이렇게 저장되면 성공: 주간날씨.csv**
```
날짜,최고기온,최저기온,강수확률
2026-11-08,14.1,5.2,10
2026-11-09,15.2,6.8,60
2026-11-10,12.4,4.9,30
2026-11-11,11.8,3.1,0
⋯ 7일치
```

**힌트** : 하루치 대신 7일치를 반복해서 꺼내기

1. 예보 기간을 7일로
```python
"forecast_days": [ ? ]
```
2. 날짜 개수만큼 반복하며 한 줄씩 만들기
```python
for i in range(len(daily["time"])):
    row = [daily["time"][i],
           daily["temperature_2m_max"][ ? ],
           ...]
```
3. 저장은 기본 사용법 ④와 같은 방법
```python
    writer.writerow(row)
```

In [ ]:
import csv
import requests

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 37.5665,         # 서울 위도
    "longitude": 126.9780,       # 서울 경도
    "daily": [                   # 받을 항목
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_probability_max"
    ],
    "timezone": "Asia/Seoul",
    "forecast_days": 7           # 1. 7일치
}
response = requests.get(url, params=params, timeout=10)
response.raise_for_status()
daily = response.json()["daily"]

with open("주간날씨.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["날짜", "최고기온", "최저기온", "강수확률"])
    # 2. 날짜 개수만큼 반복하며 한 줄씩 만들기
    for i in range(len(daily["time"])):
        row = [daily["time"][i],
               daily["temperature_2m_max"][i],
               daily["temperature_2m_min"][i],
               daily["precipitation_probability_max"][i]]
        # 3. 한 줄씩 저장
        writer.writerow(row)
        print(row)

**풀이**

- `"forecast_days": 7`로 바꾸면 리스트마다 값이 7개씩 들어옵니다.
- 예제 ①은 `[1]`(내일) 하나만 꺼냈지만, 미션은 `[i]`로 0번부터 6번까지 차례로 꺼냅니다.
- `range(len(daily["time"]))`는 날짜 개수만큼 0, 1, 2, … 6을 만들어 줍니다.
- 저장 파일은 노트북 파일과 같은 폴더에 생깁니다.

## 예제 ① AI 어시스턴트로 코드 만들기

**프롬프트 뼈대**

| 칸 | 적는 내용 |
|---|---|
| 목적 | 무엇을 하는 코드인지 |
| 입력 | 어떤 데이터를 받아서 |
| 처리 | 어떤 기준으로 처리할지 |
| 출력 | 결과를 어떻게 보여줄지 |
| 조건 | 사용할 도구, 지킬 제약 |

**예시 프롬프트**
```
[목적] 서울의 내일 날씨를 확인해 알림 문구를 출력하는 파이썬 코드를 만들어 줘.
[입력] Open-Meteo 예보 API, 서울 위도 37.5665 · 경도 126.9780
[처리] 내일 최고·최저기온과 최대 강수확률을 꺼내고, 강수확률 50% 이상이면 우산, 최저기온 10℃ 미만이면 외투 문구 출력
[출력] 날짜·기온·강수확률과 알림 문구를 출력하고, 초보자용 주석을 달아 줘.
[조건] requests만 사용. timeout=10을 넣고, 실패하면 바로 멈추게 해 줘.
```

AI가 준 코드를 아래 셀에 붙여 넣고 실행해 봅니다.

In [ ]:
# AI가 준 코드를 붙여 넣고 실행합니다.

## 연습문제에 사용할 API: Frankfurter

**Frankfurter** : 키 없이 쓰는 무료 환율 API → 오늘·기간별 환율을 받습니다

**주소 끝과 params**
```python
base = "https://api.frankfurter.dev/v1"
params = {"base": "USD", "symbols": "KRW"}
# 오늘 환율
requests.get(base + "/latest", params=params)
# 하루 환율
requests.get(base + "/2026-10-08", params=params)
# 기간 환율 (시작일..종료일)
requests.get(base + "/2026-10-08..2026-11-06",
             params=params)
```

| 주소 끝·params | 뜻 |
|---|---|
| `/latest` | 오늘 환율 |
| `/시작일..` | 시작일부터 오늘까지 |
| `"base"` | 기준 통화 (1단위) |
| `"symbols"` | 받을 통화 (쉼표로 여러 개) |

**기간 환율 응답 구조 (일부)**
```
"rates": {"2026-10-08": {"KRW": 1375.2},
          "2026-10-09": {"KRW": 1381.0}, ...}
```

**참고** : 유럽중앙은행 기준 환율이라 주말·공휴일 날짜는 빠집니다. (frankfurter.dev)

## 연습문제: AI 어시스턴트로 한 달 환율 추이 엑셀 만들기

**문제** : AI 어시스턴트로 코드를 만들어 최근 30일 원/달러 환율을 엑셀로 저장해 보세요.

**받을 데이터 (응답 일부)**
```
{
 "base": "USD",
 "rates": {
  "2026-10-08": {"KRW": 1375.2},
  "2026-10-09": {"KRW": 1381.0},
  ...
 }
}
```

**실행 후: 환율추이.xlsx**
```
날짜        USD→KRW
2026-10-08  1375.2
2026-10-09  1381.0
2026-10-12  1379.4
  ⋯
2026-11-06  1380.5
최고  1392.0
최저  1368.7
```

**조건**

| 항목 | 내용 |
|---|---|
| 주소 | api.frankfurter.dev/v1 |
| 기간 | 오늘 기준 최근 30일 |
| 통화 | USD → KRW |
| 저장 | 날짜·환율 표, 맨 아래 최고·최저 |

※ 주말·공휴일은 환율이 없어 빠집니다

**프롬프트 뼈대에 채워 넣기** : [목적] [입력] [처리] [출력] [조건] — 조건 표의 내용을 칸마다 나눠 적기

In [ ]:
import requests
from datetime import date, timedelta
from openpyxl import Workbook

# 교안: folder = r"C:\workspace\python20261103\6일차"
# 노트북에서는 노트북 파일과 같은 폴더에 저장합니다.
start = date.today() - timedelta(days=30)   # 30일 전
url = f"https://api.frankfurter.dev/v1/{start}.."  # 오늘까지
params = {"base": "USD", "symbols": "KRW"}

response = requests.get(url, params=params, timeout=10)
response.raise_for_status()
rates = response.json()["rates"]      # {날짜: {"KRW": 값}}

wb = Workbook()
ws = wb.active
ws.append(["날짜", "USD→KRW"])
for day, value in rates.items():
    ws.append([day, value["KRW"]])

krw = [v["KRW"] for v in rates.values()]   # 값만 모으기
ws.append(["최고", max(krw)])
ws.append(["최저", min(krw)])
wb.save("환율추이.xlsx")
print("저장 완료:", len(rates), "일")

### 연습문제 풀이: 코드에서 볼 점

실행 결과 (일수는 실행하는 날에 따라 달라집니다)
```
저장 완료: 21 일
```

| 코드 | 설명 |
|---|---|
| `{start}..` | 시작일부터 오늘까지 한 번에 요청 |
| `rates.items()` | 날짜와 값을 짝으로 꺼내기 |
| `max()` · `min()` | 리스트에서 최고·최저 찾기 |